In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

driver.get("http://localhost:5173/")
driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
driver.get("http://localhost:5173/login")

wait.until(EC.presence_of_element_located((By.ID, "username")))
driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
driver.find_element(By.ID, "password").send_keys("12345678")
driver.find_element(By.ID, "sign-in-btn").click()

time.sleep(3)
print("URL:", driver.current_url)
print("Page text:", driver.find_element(By.TAG_NAME, "body").text[:200])

In [ ]:
try:
    # Go to CRM, open Customer Tiers, click the first customer row to open the profile drawer
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'CRM')]"))).click()
    wait.until(EC.element_to_be_clickable((By.XPATH, "//nav[@aria-label='CRM sections']//button[contains(., 'Customer Tiers')]"))).click()
    wait.until(EC.element_to_be_clickable((By.XPATH, "(//tr[contains(@class, 'crm-tr') and contains(@class, 'is-click')])[1]"))).click()
    time.sleep(2)

    # Open the Health Information profile tab (verified in CRMModule.jsx PROFILE_TABS)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//div[@aria-label='Customer profile sections']//button[text()='Health Information']"))).click()
    time.sleep(2)

    # Read the current BP from the health grid (verified: span + strong pair)
    before_bp = driver.find_element(By.XPATH, "//span[text()='Blood Pressure']/following-sibling::strong[1]").text
    print("Before BP:", before_bp)

    # Open the editor: "Edit Health Information" if a record exists, else "Health Check"
    if [b for b in driver.find_elements(By.XPATH, "//button[contains(., 'Edit Health Information')]") if b.is_displayed()]:
        driver.find_element(By.XPATH, "//button[contains(., 'Edit Health Information')]").click()
    else:
        driver.find_element(By.XPATH, "//button[contains(., 'Health Check')]").click()
    time.sleep(2)

    # Real BP field: label "Blood Pressure", placeholder "e.g. 120/80" (verified in CRMModule.jsx)
    bp_input = wait.until(EC.visibility_of_element_located((By.XPATH, "//span[text()='Blood Pressure']/ancestor::label[1]//input")))
    bp_input.clear()
    bp_input.send_keys("120/80")

    # Save with the real modal button
    driver.find_element(By.XPATH, "//button[contains(., 'Save Health Check') or contains(., 'Save Changes')]").click()
    time.sleep(3)

    after_bp = driver.find_element(By.XPATH, "//span[text()='Blood Pressure']/following-sibling::strong[1]").text
    print("After BP:", after_bp)
    assert after_bp == "120/80", f"BP not updated (shown: {after_bp!r})."

    # Reload persistence check: refresh, then walk back to the same health grid
    driver.refresh()
    time.sleep(3)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'CRM')]"))).click()
    wait.until(EC.element_to_be_clickable((By.XPATH, "//nav[@aria-label='CRM sections']//button[contains(., 'Customer Tiers')]"))).click()
    wait.until(EC.element_to_be_clickable((By.XPATH, "(//tr[contains(@class, 'crm-tr') and contains(@class, 'is-click')])[1]"))).click()
    time.sleep(2)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//div[@aria-label='Customer profile sections']//button[text()='Health Information']"))).click()
    time.sleep(2)
    reloaded_bp = driver.find_element(By.XPATH, "//span[text()='Blood Pressure']/following-sibling::strong[1]").text
    print("BP after reload:", reloaded_bp)
    assert reloaded_bp == "120/80", "BP value did not persist after reload."

    print("Current URL:", driver.current_url)
    print("PASS: BP Update")
except Exception as e:
    print("FAIL: BP Update")
    print("Error:", e)
    driver.save_screenshot("26_bp_update_FAIL.png")

In [ ]:
driver.quit()